In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.join(os.getcwd(), "..","..")
sys.path.append(project_path)

from utils.Transformations import reuse

# **DimUser**

In [0]:
df = spark.read.format("parquet")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimUser")

In [0]:
display(df)

## Auto Loader

In [0]:
df_user = spark.readStream.format("cloudFiles")\
               .option("cloudFiles.format", "parquet")\
               .option("cloudFiles.schemaLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimUser/Checkpoint")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimUser")

In [0]:
df_user.isStreaming

In [0]:
df_user= df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
df_user_obj = reuse()
df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])


In [0]:
df_user.isStreaming

In [0]:
display(df_user, checkpointLocation="abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimUser/Checkpoint")

In [0]:
df_user.writeStream.format("delta")\
                   .option("checkpointLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimUser/Checkpoint")\
                   .trigger(once=True)\
                   .option("path", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimUser/Data")\
                   .toTable("catalog_azure.silver.DimUser")

#DimArtist

In [0]:
df = spark.read.format("parquet")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimArtist")

In [0]:
display(df)

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
               .option("cloudFiles.format", "parquet")\
               .option("cloudFiles.schemaLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimArtist/Checkpoint")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimArtist")

In [0]:
df_artist.isStreaming

In [0]:
df_artist= df_artist.withColumn("artist_name",upper(col("artist_name")))

In [0]:
df_artist_obj = reuse()
df_artist = df_artist_obj.dropColumns(df_artist,['_rescued_data'])


In [0]:
df_artist.writeStream.format("delta")\
                   .option("checkpointLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimArtist/Checkpoint")\
                   .trigger(once=True)\
                   .option("path", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimArtist/Data")\
                   .toTable("catalog_azure.silver.DimArtist")

#DimTrack

In [0]:
df = spark.read.format("parquet")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimTrack")

In [0]:
display(df)

In [0]:
df_track = spark.readStream.format("cloudFiles")\
               .option("cloudFiles.format", "parquet")\
               .option("cloudFiles.schemaLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimTrack/Checkpoint")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/DimTrack")

In [0]:
df_track.isStreaming

In [0]:
df_track= df_track.withColumn("track_name",upper(col("track_name")))

In [0]:
df_track_obj = reuse()
df_track = df_track_obj.dropColumns(df_track,['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])

In [0]:
df_track.writeStream.format("delta")\
                   .option("checkpointLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimTrack/Checkpoint")\
                   .trigger(once=True)\
                   .option("path", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/DimTrack/Data")\
                   .toTable("catalog_azure.silver.DimTrack")

#FactStream

In [0]:
df = spark.read.format("parquet")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/FactStream")

In [0]:
display(df)

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
               .option("cloudFiles.format", "parquet")\
               .option("cloudFiles.schemaLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/FactStream/Checkpoint")\
               .load("abfss://bronze@azureprojectstoragegroup.dfs.core.windows.net/FactStream")

In [0]:
df_fact.isStreaming

In [0]:
df_fact_obj = reuse()
df_fact = df_fact_obj.dropColumns(df_fact,['_rescued_data'])
df_fact = df_fact.dropDuplicates(['user_id'])

In [0]:
df_fact.writeStream.format("delta")\
                   .option("checkpointLocation", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/FactStream/Checkpoint")\
                   .trigger(once=True)\
                   .option("path", "abfss://silver@azureprojectstoragegroup.dfs.core.windows.net/FactStream/Data")\
                   .toTable("catalog_azure.silver.FactStream")